Import the necessary libraries

In [ ]:
!pip install -U optimum[onnxruntime] transformers torch torchvision onnxruntime pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 16.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.4.0
    Uninstalling transformers-5.4.0:
      Successfully uninstalled transformers-5.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatible.


In [ ]:
from optimum.onnxruntime import ORTModelForImageClassification
from transformers import AutoImageProcessor

model_id = "nateraw/food"

model = ORTModelForImageClassification.from_pretrained(
    model_id,
    export=True
)

processor = AutoImageProcessor.from_pretrained(model_id)

model.save_pretrained("onnx_model")
processor.save_pretrained("onnx_model")

Multiple distributions found for package optimum. Picked distribution: optimum-onnx
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
Using a slow image processor as `use_fast` is unse

['onnx_model/preprocessor_config.json']

In [ ]:
!ls onnx_model

config.json  model.onnx  preprocessor_config.json


In [ ]:
import json

labels = model.config.id2label

with open("labels.json", "w") as f:
    json.dump(labels, f, indent=2)

print("✅ labels.json created")

✅ labels.json created


In [ ]:
import onnxruntime as ort
from PIL import Image
import numpy as np

# Load ONNX model
session = ort.InferenceSession("onnx_model/model.onnx")

# Get correct input name (IMPORTANT)
input_name = session.get_inputs()[0].name
print("Input name:", input_name)

# Load image
image = Image.open("test1.jpg").convert("RGB")

# Preprocess (VERY IMPORTANT)
inputs = processor(images=image, return_tensors="np")

# Run inference
outputs = session.run(None, {input_name: inputs["pixel_values"]})

# Get prediction
logits = outputs[0]
predicted_class = int(np.argmax(logits))

# Load labels
with open("labels.json") as f:
    label_map = json.load(f)

print("🍔 Prediction:", label_map[str(predicted_class)])

Input name: pixel_values
🍔 Prediction: lasagna


In [ ]:
print("Input shape:", session.get_inputs()[0].shape)
print("Output shape:", session.get_outputs()[0].shape)

Input shape: ['batch_size', 'num_channels', 'height', 'width']
Output shape: ['batch_size', 101]


In [ ]:
len(labels)

101

**FP16 Conversion**

In [ ]:
!pip install onnxconverter-common

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 3.1 MB/s eta 0:00:00


In [ ]:
from onnxconverter_common import float16
import onnx

# Load original model
model = onnx.load("onnx_model/model.onnx")

# Convert to FP16
model_fp16 = float16.convert_float_to_float16(model)

# Save new model
onnx.save(model_fp16, "model_fp16.onnx")

print("✅ FP16 model created")

/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 4.2559307189549145e-08 will be truncated to 1e-07
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -3.782829605114557e-09 will be truncated to -1e-07
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 4.4152947964448686e-08 will be truncated to 1e-07
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 6.0694922865423e-08 will be truncated to 1e-07
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 8.560190423168024e-08 will be truncated to 1e-07
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -1.4117072844044287e

✅ FP16 model created


In [ ]:
import onnxruntime as ort
from PIL import Image
import numpy as np

# Load ONNX model
session = ort.InferenceSession("model_fp16.onnx")

# Get correct input name (IMPORTANT)
input_name = session.get_inputs()[0].name
print("Input name:", input_name)

# Load image
image = Image.open("test.jpg").convert("RGB")

# Preprocess (VERY IMPORTANT)
inputs = processor(images=image, return_tensors="np")

# 🔥 Convert to float16 (IMPORTANT)
inputs["pixel_values"] = inputs["pixel_values"].astype(np.float16)

# Run inference
outputs = session.run(None, {input_name: inputs["pixel_values"]})

# Get prediction
logits = outputs[0]
predicted_class = int(np.argmax(logits))

# Load labels
with open("labels.json") as f:
    label_map = json.load(f)

print("🍔 Prediction:", label_map[str(predicted_class)])

Input name: pixel_values
🍔 Prediction: pizza


**INT8** **Qunatization**

In [ ]:
!pip install onnxruntime-tools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.7/212.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.7 MB/s eta 0:00:00


In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    "onnx_model/model.onnx",
    "model_int8.onnx",
    weight_type=QuantType.QInt8
)

print("✅ INT8 model created")

✅ INT8 model created


In [ ]:
import onnxruntime as ort
from PIL import Image
import numpy as np

# Load ONNX model
session = ort.InferenceSession("model_int8.onnx")

# Get correct input name (IMPORTANT)
input_name = session.get_inputs()[0].name
print("Input name:", input_name)

# Load image
image = Image.open("test1.jpg").convert("RGB")

# Preprocess (VERY IMPORTANT)
inputs = processor(images=image, return_tensors="np")

# Run inference
outputs = session.run(None, {input_name: inputs["pixel_values"]})

# Get prediction
logits = outputs[0]
predicted_class = int(np.argmax(logits))

# Load labels
with open("labels.json") as f:
    label_map = json.load(f)

print("🍔 Prediction:", label_map[str(predicted_class)])

Input name: pixel_values
🍔 Prediction: lasagna
